# 日本語感情音声認識入門: JNV corpus と MFCC-LSTM

この notebook では、公式配布されている **JNV: Japanese Nonverbal Vocalization corpus** を使い、音声から感情を分類する基本的な流れを学びます。主な目的は、精度の高いモデルを作ることではなく、音声がどのような数値表現に変換され、その数値をどのように機械学習モデルへ入力するのかを理解することです。

本 notebook で扱う内容は次の通りです。

- 音声波形、スペクトログラム、Mel スペクトログラム、MFCC の定義と可視化
- ピッチ、基本周波数 F0、フォルマントの定義と Praat による可視化
- MFCC 特徴量の PCA / LDA による2次元表示
- MFCC の時系列を PyTorch の LSTM に入力する感情分類

使用するデータセットは次の通りです。

- 公式ページ: https://sites.google.com/site/shinnosuketakamichi/research-topics/jnv_corpus
- 公式 zip: https://ss-takashi.sakura.ne.jp/corpus/jnv/jnv_corpus_ver3.zip
- ライセンス: CC BY-SA 4.0
- 感情ラベル: angry, disgust, fear, happy, sad, surprise
- 話者: F1, F2, M1, M2

JNV は通常の文章読み上げではなく、日本語の非言語発声、たとえば笑い、泣き、叫びに近い発声を含む感情音声コーパスです。身近な例で言えば、「同じ言葉を言っていなくても、声の高さ、強さ、揺れ方から感情が伝わる」状況を扱います。


In [ ]:
!pip -q install librosa soundfile scikit-learn pandas matplotlib seaborn praat-parselmouth torch

from pathlib import Path
import sys

REPO_URL = "https://github.com/akio-kobayashi/EmotionRecog.git"  # 必要なら自分のGitHub URLに変更する
REPO_DIR = Path("/content/EmotionRecog")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

sys.path.insert(0, str(REPO_DIR))
from ser import *

DATA_ROOT = Path("/content/data")
EVAL_MODE = "random"  # 発展: "speaker_holdout" にすると未知話者評価になる


## JNV のダウンロードと展開

**コーパス**とは、研究や学習に使うために一定の基準で集められたデータ集です。音声認識や感情認識では、音声ファイルと、それに対応するラベルをそろえたコーパスを使います。

このセルでは、JNV の公式 zip ファイルを Colab 上にダウンロードし、wav ファイルを展開します。自分で録音した音声ではなく公開コーパスを使うことで、他の人も同じ条件で実験を再現できます。


In [ ]:
wav_paths, jnv_dir = download_jnv(DATA_ROOT)
print(f"wav files: {len(wav_paths)}")
print(wav_paths[:5])


## manifest の生成

**manifest**とは、データセットに含まれる各ファイルについて、ファイルパス、ラベル、話者などの情報を表にしたものです。機械学習では、音声ファイルを直接眺めるだけでなく、「このファイルはどの感情か」「誰の発声か」を表形式で管理する必要があります。

JNV の wav ファイル名は `M1_happy_01_F.wav` のような形式です。この名前から、話者ID、感情ラベル、発話番号、セッション種別を取り出します。たとえば `M1_happy_01_F.wav` は、話者 `M1` による `happy` の発声であることを表しています。`R` は regular session、`F` は phrase-free session です。


In [ ]:
df = build_manifest(wav_paths, DATA_ROOT / "jnv_manifest.csv")

display(df.head())
display(pd.crosstab(df["label"], df["speaker_id"]))
display(pd.crosstab(df["label"], df["session"]))


## 波形・スペクトログラム・MFCC の確認

音声は、空気の圧力変化が時間に沿って並んだ信号です。コンピュータでは、この連続的な変化を一定間隔で測った数列として扱います。ここでは、1つの音声を題材にして、音声処理でよく使う表現を順に見ます。

**波形**とは、横軸を時間、縦軸を振幅として、音の変化をそのまま描いたものです。振幅は空気の振動の大きさに対応します。手をたたいた音なら短い時間に大きな振幅が現れ、長く伸ばした声なら振幅がある程度続きます。

**スペクトログラム**とは、音声を短い時間区間に分け、それぞれの区間にどの周波数成分がどれくらい含まれるかを色で表した図です。横軸は時間、縦軸は周波数、色は強さを表します。楽譜が「いつ、どの高さの音が鳴るか」を表すのに似ていますが、スペクトログラムはより細かい周波数成分まで表示します。

**Mel スペクトログラム**とは、周波数軸を人間の聴覚特性に近い Mel 尺度で表したスペクトログラムです。人間は低い周波数の違いには比較的敏感で、高い周波数の細かい違いには鈍感です。Mel 尺度は、この聞こえ方の違いを反映した周波数尺度です。

**MFCC**、Mel-Frequency Cepstral Coefficients とは、Mel スペクトログラムを対数化し、DCT、離散コサイン変換によって少数の係数にまとめた特徴量です。直観的には、声の音色や声道の形に関係する「なめらかなスペクトルの形」を、扱いやすい数値列として取り出す方法です。


In [ ]:
demo_label = "happy"  # 例: angry, disgust, fear, happy, sad, surprise
demo_row, y_demo, sr_demo = load_demo_audio(df, label=demo_label)
demo_path = demo_row["path"]

print(demo_row.to_dict())
print(f"samples: {len(y_demo)}, sampling rate: {sr_demo} Hz, duration: {len(y_demo) / sr_demo:.2f} s")

display(Audio(y_demo, rate=sr_demo))
plot_waveform(y_demo, sr_demo)
plot_spectrogram(y_demo, sr_demo)
mel_db = plot_mel_spectrogram(y_demo, sr_demo)


音の強さは、物理的には非常に広い範囲の値を取ります。そのため、MFCC では Mel スペクトログラムのパワーを **対数化**します。対数化とは、大きすぎる差を圧縮して扱いやすくする変換です。身近な例では、地震のマグニチュードや音の dB 表示も、非常に広い範囲の量を人間が扱いやすい尺度に直す考え方です。

次に、対数化した Mel スペクトルに **DCT**、離散コサイン変換をかけます。DCT は、並んだ数値を「ゆっくり変化する成分」と「細かく変化する成分」に分ける変換です。FFT が時間波形を周波数成分に分解するのと似ていますが、MFCC では周波数方向に並んだ log-Mel スペクトルの形を分解します。

ここで重要なのが、**スペクトル包絡**と**微細構造**の違いです。スペクトル包絡とは、スペクトル全体をなめらかに結んだ大まかな形です。これは口の開き方、舌の位置、唇の丸めなど、声道形状や調音の違いを反映します。たとえば「あ」と「い」では、口や舌の形が違うため、スペクトル包絡も変わります。

一方、微細構造とは、スペクトル上に細かく現れる山や谷の並びです。これは声帯振動の周期性、基本周波数 F0、倍音構造と関係します。たとえば同じ母音でも、低い声と高い声では倍音の間隔が変わります。

MFCC の低次係数は主にスペクトル包絡を表しやすく、高次係数はより細かな構造を含みやすいと考えられます。音声認識では声道や音色に関係する情報を取り出すために MFCC がよく使われます。ただし感情音声では、声の高さや抑揚も重要なので、MFCC だけですべてを説明できるわけではありません。

MFCC をそのままヒートマップにすると、係数ごとの値の範囲がそろっていないため、色の違いが見えにくくなることがあります。特に C0 は全体のエネルギーに近い情報を強く持つため、色スケールを支配しやすい係数です。そこで、この notebook の MFCC 図では C0 を除き、C1 以降を係数ごとに標準化して表示します。これは絶対値を見る図ではなく、「各係数がその発話の中でいつ大きくなり、いつ小さくなるか」を見るための図です。下段には、低次係数の時間変化を折れ線として表示します。


In [ ]:
plot_log_mel_dct_frame(mel_db)
mfcc_demo = plot_mfcc(y_demo, sr_demo)


ここでは、後で PCA や LDA によって特徴量を2次元表示するために、MFCC の各係数について平均と標準偏差を計算します。これは、時間とともに変化する MFCC を、1つの発話につき固定長のベクトルへ要約する操作です。

ただし、この要約では「いつ声が高くなったか」「どこで急に強くなったか」といった時間方向の情報が失われます。そのため、この notebook の分類モデルでは、後半で MFCC の時系列そのものを LSTM に入力します。平均と標準偏差は、まず特徴量の全体像を眺めるための入門的な表現として使います。


In [ ]:
mfcc_summary = display_mfcc_summary(mfcc_demo)


## ピッチ・基本周波数の確認

**ピッチ**とは、人間が感じる音の高さです。音響学では、周期的な音声の高さに対応する物理量として **基本周波数 F0** を用います。F0 は、声帯振動が1秒間に何回繰り返されるかを表す周波数です。単位は Hz です。

たとえば F0 が 200 Hz なら、声帯が1秒間に約200回振動していることを意味します。一般に、F0 が高いほど高い声として聞こえ、F0 が低いほど低い声として聞こえます。

感情音声認識では、ピッチは重要な手がかりです。驚きや怒りでは声が高くなったり急に変化したりしやすく、悲しみでは低く平坦になりやすいことがあります。もちろん個人差はありますが、「楽しそうな声はよく上下し、元気のない声は単調に聞こえる」という日常的な感覚と対応しています。


In [ ]:
pitch_stats = draw_waveform_with_pitch(demo_path)
display(pd.DataFrame([pitch_stats]).round(2))

pitch_stats_df = summarize_pitch_by_label(df)


## Praat によるスペクトログラム・フォルマント描画

**フォルマント**とは、声道で共鳴して強く現れる周波数帯のことです。音声学では、第1フォルマント F1、第2フォルマント F2、第3フォルマント F3 などとして表します。フォルマントは、母音の違いや口の形を表す重要な手がかりです。

たとえば、同じ高さの声でも「あ」と「い」が違って聞こえるのは、声帯だけでなく、口や舌の形によって共鳴のしかたが変わるからです。この共鳴の違いがフォルマントとして観察されます。

ここでは `praat-parselmouth` を使い、Praat の方法で推定したフォルマント軌跡をスペクトログラム上に重ねます。JNV は非言語発声を含むため、母音のように常に安定したフォルマントが出るとは限りません。したがって、ここでは厳密な音声学的分析というより、感情ラベルごとの音響的な違いを観察する教材として使います。

**注意:** この図の F1、F2、F3 は Praat が各時刻で推定した値です。処理の都合上、無音区間、無声区間、雑音的な区間でも点が表示されることがあります。したがって、点が描かれているからといって、その時刻に母音のような安定した声道共鳴が実際に存在するとは限りません。図を読むときは、背景のスペクトログラムでエネルギーが強い有声らしい区間と重なっているかを確認してください。


In [ ]:
target_label = "happy"  # 例: angry, disgust, fear, happy, sad, surprise
sample_row = df[df["label"] == target_label].iloc[0]
print(sample_row.to_dict())
draw_spectrogram_with_formants(sample_row["path"])

formant_stats = summarize_formants_by_label(df)


## PCA/LDA 用の固定長特徴量抽出

**特徴量**とは、機械学習モデルに入力するためにデータから取り出した数値です。音声そのものは長い波形ですが、そのままでは扱いにくいため、MFCC のような特徴量に変換します。

この節では、wav ファイルを 16 kHz の mono 音声として読み込み、MFCC、ΔMFCC、ΔΔMFCC を計算します。ΔMFCC は MFCC の時間変化、ΔΔMFCC はその変化のさらに変化を表します。車の動きにたとえると、MFCC が位置、ΔMFCC が速度、ΔΔMFCC が加速度に近い役割を持ちます。

ここでは PCA/LDA の可視化のために、各係数の平均と標準偏差を結合して固定長ベクトルを作ります。分類モデルである LSTM では、この後で平均化していない MFCC 時系列を使います。


In [ ]:
X, y, groups, label_encoder = build_mfcc_stat_dataset(df)
print("X:", X.shape)
print("labels:", list(label_encoder.classes_))


## MFCC 特徴量の PCA と LDA 可視化

この節では、MFCC から作った固定長特徴量を2次元に写して、感情ラベルや話者差がどのように見えるかを確認します。高次元のデータを2次元に写すと、完全な情報は失われますが、人間が目で全体の傾向をつかみやすくなります。

**主成分分析 PCA**、Principal Component Analysis とは、多数の特徴量を、データのばらつきが大きい少数の軸に変換する方法です。第1主成分 PC1 はデータが最も大きく広がる方向、第2主成分 PC2 は PC1 と直交し、次に大きく広がる方向です。

直観的には、机の上に散らばった点を、最も横長に見える方向から写真に撮るようなものです。PCA は「データの違いがよく見える向き」を探します。ただし、PCA は感情ラベルを使いません。そのため、PCA の軸は「感情を分ける方向」ではなく、「データ全体のばらつきが大きい方向」です。

ここで注意が必要です。PCA の散布図に外れ値があると、軸の表示範囲が外れ値に引っ張られ、ほとんどの点が細い帯のようにつぶれて見えることがあります。その図を見ると、PC2 方向には情報がないように誤解しやすくなります。

そこで、この notebook の PCA 図では、PCA の計算自体は全サンプルで行った上で、描画では中心部分のサンプルだけを表示し、PC2 方向を少し拡大して見せます。これは「PCAを都合よく変えた」のではなく、外れ値で図が読みにくくなることを避けるための表示上の工夫です。図のタイトルには、表示しているサンプル数を `central view` として示します。

したがって、PCA で感情がきれいに分かれないことは失敗ではありません。話者差、音量差、録音条件、発声の個人差の方が大きければ、PCA の散布図では感情よりもそれらの違いが強く見えます。


In [ ]:
X_scaled, pca_df, pca = plot_pca_mfcc(df, X)


PCA で感情があまり分離しない場合は、その結果自体が重要な観察です。PCA は教師なしの方法なので、感情ラベルを知らずに軸を決めます。つまり、PCA は「感情を当てるための最適な軸」を探しているわけではありません。

比較のため、次に **LDA**、Linear Discriminant Analysis、線形判別分析を使います。LDA はラベルを使う教師ありの方法です。クラス間の距離が大きく、同じクラス内のばらつきが小さくなるような軸を探します。

直観的には、PCA が「点群全体がよく広がって見える向き」を探すのに対し、LDA は「感情ラベルごとに分けて見やすい向き」を探します。ただし、LDA の図はラベルを使って作られた可視化です。そのため、図で分かれて見えても、未知の音声に対する分類性能が高いことを直接保証するものではありません。


In [ ]:
lda_df, lda = plot_lda_mfcc(df, X_scaled, y, label_encoder)
plot_pca_by_speaker(pca_df)


## MFCC 時系列を使う LSTM

ここからは、MFCC を平均や標準偏差にまとめず、時間方向に並んだ系列として扱います。音声は時間とともに変化するデータです。したがって、感情を考えるときには、ある瞬間の音色だけでなく、声の立ち上がり、揺れ、持続、終わり方も重要になります。

**LSTM**、Long Short-Term Memory とは、系列データを扱うための再帰型ニューラルネットワークの一種です。現在の入力だけでなく、それ以前の入力から受け継いだ内部状態を使って予測します。普通のニューラルネットワークが1枚の写真を見るモデルだとすれば、LSTM は短い動画を順番に見て内容を判断するモデルに近いです。

ここでは、まず `random` split を既定値にします。これは「MFCC 時系列から感情ラベルを学習できるか」を確認するための基本設定です。`speaker_holdout` は未知話者に対する評価で、4話者しかない JNV では非常に厳しく、1つの感情に予測が潰れることがあります。その場合は、モデルが感情を学習したとは言えません。

各時刻の入力として MFCC、ΔMFCC、ΔΔMFCC を使います。これにより、音色そのものだけでなく、音色がどのように変化しているかもモデルに渡します。発話ごとに長さが違うため、短い系列には padding を入れますが、`pack_padded_sequence` を使って LSTM が padding 部分を学習しないようにします。


In [ ]:
train_idx, test_idx = make_split(df, y, groups, mode=EVAL_MODE)
display_split_summary(df, y, groups, label_encoder, train_idx, test_idx, EVAL_MODE)


音声ファイルごとに長さが違うため、そのままでは同じミニバッチにまとめられません。そこで、短い系列の末尾に 0 を追加して長さをそろえます。この操作を **padding** と呼びます。

ただし、padding は本物の音声ではありません。そのため、標準化の平均と分散は、padding を含まない実際の音声フレームだけから推定します。また、標準化のパラメータは学習データだけで計算し、テストデータの情報が学習手順に混ざらないようにします。これは、試験問題を事前に見てから勉強方法を決めない、という考え方に似ています。


In [ ]:
X_seq, seq_lengths, n_features, inner_train_idx, val_idx, frame_scaler = build_sequence_dataset(
    df,
    train_idx,
    y,
)


LSTM には、padding 済みの系列と、本来のフレーム長を一緒に渡します。`pack_padded_sequence` は、系列の本当の長さを使って padding 部分を無視するための PyTorch の仕組みです。

この notebook では **双方向 LSTM** を使います。通常の LSTM は発話の前から後ろへ順に読みますが、双方向 LSTM は後ろから前へ読む流れも同時に使います。さらに、最後の隠れ状態だけで判断するのではなく、LSTM が各時刻で出力した表現を発話全体にわたって pooling します。

これは重要です。感情は最後の一瞬だけで決まるとは限りません。たとえば、最初に急に高くなった声、途中で震える声、最後まで強く続く声など、発話全体に分散した手がかりがあります。そこで padding 部分を除いた mean pooling と max pooling を使い、発話全体の情報をまとめます。


In [ ]:
lstm_model, history_df, device = train_lstm(
    X_seq,
    seq_lengths,
    y,
    inner_train_idx,
    val_idx,
    label_encoder,
)


In [ ]:
lstm_pred, lstm_result, prediction_summary, cm_lstm = evaluate_lstm(
    lstm_model,
    X_seq,
    seq_lengths,
    y,
    train_idx,
    test_idx,
    label_encoder,
    device,
)


## 実行後の結果の見方

評価セルでは、majority baseline、LSTM の accuracy / macro F1、予測ラベル分布、classification report、混同行列が表示されます。ここでは、数値を1つだけ見て判断せず、複数の出力を合わせて読みます。

まず **majority baseline** を確認します。これは、学習データで最も多いラベルだけを常に予測する単純な基準です。LSTM の accuracy や macro F1 がこの基準とほとんど変わらない場合、モデルはあまり意味のある分類をしていない可能性があります。

次に **予測ラベル分布**を見ます。`predicted_count` が1つ、または2つの感情に極端に集中している場合、モデルは多くの入力を同じ感情として扱っています。たとえば、ほとんどを `surprise` と予測しているなら、混同行列の `surprise` 列だけが濃くなります。この場合は、accuracy が少し高くても、感情の違いを学習したとは言えません。

最後に **混同行列**を読みます。縦軸が正解ラベル、横軸が予測ラベルです。対角線上の値は正解数を表します。対角線が濃いほど、その感情を正しく分類できています。対角線以外に大きな値がある場合は、どの感情をどの感情と間違えたかを表します。

読むときのヒントは次の通りです。

- 対角線に値がある: その感情をある程度識別できている。
- 1つの列だけが濃い: モデルが同じラベルばかり予測しており、失敗に近い。
- `angry` と `happy`、`sad` と `angry` などが混ざる: MFCC 時系列だけでは声の強さや抑揚の似た感情を分けにくい可能性がある。
- `surprise` が比較的当たりやすい: 立ち上がりや高さの変化など、音響的に目立つ特徴が出ている可能性がある。
- `disgust` や `fear` が弱い: ラベルの音響的な境界が曖昧、またはデータ数・話者数が足りない可能性がある。

したがって、今回のモデルは「実用的な感情認識器」としてではなく、「MFCC 時系列を LSTM に入れると、どの感情が見えやすく、どの感情が混ざりやすいかを観察する教材」として読みます。特に、混同行列が1列に潰れていないか、majority baseline を上回っているか、どの感情ペアで誤分類が多いかを確認してください。


## 次の拡張

この notebook では、初学者向けに最小限の構成で感情音声認識を扱いました。発展課題として、次のような拡張が考えられます。

- 4 話者それぞれを test speaker にする leave-one-speaker-out 評価を行い、話者が変わったときの難しさを調べる。
- F0 や energy の時系列特徴量を MFCC と結合し、LSTM に入力する。
- `R` と `F` の session 別に分類の難しさを比較する。
- LSTM でも難しい場合は、Wav2Vec2/HuBERT 系の事前学習済み音声表現を使う。

まずは、この notebook の範囲で「音声を数値に変換する」「特徴量を可視化する」「時系列モデルで分類する」という3つの流れを押さえることを目標にします。
